In [ ]:
!uv pip install -q gensim ufal.udpipe

import re

import gensim
import nltk
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ufal.udpipe import Model, Pipeline

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.8/953.8 kB 21.9 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!wget 'https://rusvectores.org/static/models/udpipe_syntagrus.model'
!wget 'https://vectors.nlpl.eu/repository/20/220.zip'
!wget 'https://vectors.nlpl.eu/repository/20/213.zip'

!unzip '/content/220.zip' -d "/content/w2v"
!unzip '/content/213.zip' -d "/content/ft"

--2026-03-18 18:37:22--  https://rusvectores.org/static/models/udpipe_syntagrus.model
Resolving rusvectores.org (rusvectores.org)... 129.240.189.200, 2001:700:112::200
Connecting to rusvectores.org (rusvectores.org)|129.240.189.200|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40616122 (39M)
Saving to: ‘udpipe_syntagrus.model’

udpipe_syntagrus.mo 100%[===================>]  38.73M  17.9MB/s    in 2.2s    

2026-03-18 18:37:25 (17.9 MB/s) - ‘udpipe_syntagrus.model’ saved [40616122/40616122]

--2026-03-18 18:37:25--  https://vectors.nlpl.eu/repository/20/220.zip
Resolving vectors.nlpl.eu (vectors.nlpl.eu)... 129.240.189.200, 2001:700:112::200
Connecting to vectors.nlpl.eu (vectors.nlpl.eu)|129.240.189.200|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 638171816 (609M) [application/zip]
Saving to: ‘220.zip’

220.zip             100%[===================>] 608.61M  27.5MB/s    in 23s     

2026-03-18 18:37:49 (26.2 MB/s) - ‘220.zip’ sa

# 1. Предобработка из первой работы

In [ ]:
pos = pd.read_csv("/content/positive.csv", sep=";", header=None)
neg = pd.read_csv("/content/negative.csv", sep=";", header=None)
tweet = pd.concat([pos, neg])
sentences = tweet[3].to_list()

In [ ]:
with open("/content/your_file.txt") as file:
    preprocessed = file.read().splitlines()
preprocessed[0]

'и школота поверь нас самое ахах общество профилирующий предмет'

# 2. Обучение W2V модели

In [ ]:
w2v = gensim.models.Word2Vec(preprocessed, vector_size=300, min_count=2, window=3)

In [ ]:
w2v.wv.get_vector("и")

array([-0.05355097, -0.12302762, -0.34865394, -0.09351049,  0.11552963,
        0.01774398,  0.04593998, -0.09749857,  0.10141207,  0.20684972,
       -0.23580149, -0.0171317 ,  0.06215768, -0.2386491 ,  0.10665379,
       -0.12452793,  0.03496749,  0.00378228,  0.08984128,  0.20380743,
        0.3122718 , -0.09270401, -0.22596504,  0.19249134, -0.16834621,
        0.1244432 , -0.30387542,  0.28514   ,  0.01725048, -0.19926424,
        0.04187225, -0.34696373,  0.03200623,  0.03028656, -0.04829941,
       -0.0083455 ,  0.2683596 , -0.15013552, -0.18360372,  0.03598712,
        0.17888878, -0.04957451,  0.08902773,  0.05640731,  0.09221905,
        0.2911652 , -0.16954333,  0.15146379,  0.25111926,  0.03173811,
        0.03264917, -0.20467764,  0.1926213 , -0.15879792, -0.21727096,
        0.04000653, -0.22151439,  0.04139236,  0.09458739, -0.06792146,
       -0.02783093, -0.13141489, -0.1627381 ,  0.00190693, -0.02949022,
       -0.01306045,  0.0792551 ,  0.10146935, -0.16087793,  0.05

Загружаем RusVectors

In [ ]:
w2v_rus = gensim.models.KeyedVectors.load_word2vec_format("/content/w2v/model.bin", binary=True)
w2v_rus.get_vector("день_NOUN")

array([ 1.607006  ,  0.88105875, -0.90222335,  0.1547316 ,  2.5412288 ,
       -0.7520914 ,  1.7317989 ,  0.5731876 , -2.1542902 , -2.353275  ,
        0.95301574,  3.7874365 , -0.17544061,  2.6331015 ,  1.0433667 ,
       -0.03230344, -2.2424862 , -1.4904966 , -1.6885313 ,  4.0017266 ,
       -1.4873472 ,  1.4159247 ,  0.926376  , -1.4879751 , -1.5259216 ,
        0.3665371 , -1.9313074 ,  3.8923445 , -4.37349   ,  2.7896214 ,
        0.7970378 ,  0.1995172 ,  2.0809715 ,  3.2362456 ,  0.28836358,
        0.12927732, -1.1088953 , -3.2749522 , -1.3922808 , -0.1875049 ,
        2.2316453 , -0.5616649 ,  1.2011256 ,  3.290491  ,  3.4515133 ,
        3.8490689 ,  0.4659025 ,  2.321132  , -0.4208103 ,  4.1541185 ,
       -1.7787856 , -2.098688  , -1.7629033 ,  0.6427573 ,  0.6830237 ,
        3.0742538 ,  2.8488643 ,  3.6958883 ,  0.8942978 , -1.3142806 ,
        1.2748865 , -0.37396657, -6.501099  ,  3.2598054 ,  4.228989  ,
        0.0256408 ,  2.6925178 ,  1.7257009 , -2.8580184 , -1.58

Код предобработки

In [ ]:
def num_replace(word):
    newtoken = "x" * len(word)
    return newtoken


def clean_token(token, misc):
    """
    :param token:  токен (строка)
    :param misc:  содержимое поля "MISC" в CONLLU (строка)
    :return: очищенный токен (строка)
    """
    out_token = token.strip().replace(" ", "")
    if token == "Файл" and "SpaceAfter=No" in misc:
        return None
    return out_token


def list_replace(search, replacement, text):
    search = [el for el in search if el in text]
    for c in search:
        text = text.replace(c, replacement)
    return text


def unify_sym(text):  # принимает строку в юникоде
    text = list_replace(
        "\u00ab\u00bb\u2039\u203a\u201e\u201a\u201c\u201f\u2018\u201b\u201d\u2019",
        "\u0022",
        text,
    )

    text = list_replace("\u2012\u2013\u2014\u2015\u203e\u0305\u00af", "\u2003\u002d\u002d\u2003", text)

    text = list_replace("\u2010\u2011", "\u002d", text)

    text = list_replace(
        "\u2000\u2001\u2002\u2004\u2005\u2006\u2007\u2008\u2009\u200a\u200b\u202f\u205f\u2060\u3000",
        "\u2002",
        text,
    )

    text = re.sub("\u2003\u2003", "\u2003", text)
    text = re.sub("\t\t", "\t", text)

    text = list_replace(
        "\u02cc\u0307\u0323\u2022\u2023\u2043\u204c\u204d\u2219\u25e6\u00b7\u00d7\u22c5\u2219\u2062",
        ".",
        text,
    )

    text = list_replace("\u2217", "\u002a", text)

    text = list_replace("…", "...", text)

    text = list_replace("\u2241\u224b\u2e2f\u0483", "\u223d", text)

    text = list_replace("\u00c4", "A", text)  # латинская
    text = list_replace("\u00e4", "a", text)
    text = list_replace("\u00cb", "E", text)
    text = list_replace("\u00eb", "e", text)
    text = list_replace("\u1e26", "H", text)
    text = list_replace("\u1e27", "h", text)
    text = list_replace("\u00cf", "I", text)
    text = list_replace("\u00ef", "i", text)
    text = list_replace("\u00d6", "O", text)
    text = list_replace("\u00f6", "o", text)
    text = list_replace("\u00dc", "U", text)
    text = list_replace("\u00fc", "u", text)
    text = list_replace("\u0178", "Y", text)
    text = list_replace("\u00ff", "y", text)
    text = list_replace("\u00df", "s", text)
    text = list_replace("\u1e9e", "S", text)

    currencies = list(
        "\u20bd\u0024\u00a3\u20a4\u20ac\u20aa\u2133\u20be\u00a2\u058f\u0bf9\u20bc\u20a1\u20a0\u20b4\u20a7\u20b0\u20bf\u20a3\u060b\u0e3f\u20a9\u20b4\u20b2\u0192\u20ab\u00a5\u20ad\u20a1\u20ba\u20a6\u20b1\ufdfc\u17db\u20b9\u20a8\u20b5\u09f3\u20b8\u20ae\u0192"
    )

    alphabet = list(
        "\t\n\r абвгдеёзжийклмнопрстуфхцчшщьыъэюяАБВГДЕЁЗЖИЙКЛМНОПРСТУФХЦЧШЩЬЫЪЭЮЯ,.[]{}()=+-−*&^%$#@!?~;:0123456789§/"
        '\\|"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ '
    )

    alphabet.append("'")

    allowed = set(currencies + alphabet)

    cleaned_text = [sym for sym in text if sym in allowed]
    cleaned_text = "".join(cleaned_text)

    return cleaned_text


def clean_lemma(lemma, pos):
    """
    :param lemma: лемма (строка)
    :param pos: часть речи (строка)
    :return: очищенная лемма (строка)
    """
    out_lemma = lemma.strip().replace(" ", "").replace("_", "").lower()
    if "|" in out_lemma or out_lemma.endswith(".jpg") or out_lemma.endswith(".png"):
        return None
    if pos != "PUNCT":
        if out_lemma.startswith("«") or out_lemma.startswith("»"):
            out_lemma = "".join(out_lemma[1:])
        if out_lemma.endswith("«") or out_lemma.endswith("»"):
            out_lemma = "".join(out_lemma[:-1])
        if out_lemma.endswith("!") or out_lemma.endswith("?") or out_lemma.endswith(",") or out_lemma.endswith("."):
            out_lemma = "".join(out_lemma[:-1])
    return out_lemma


def process(pipeline, text="Строка", keep_pos=True, keep_punct=False):
    # Если частеречные тэги не нужны (например, их нет в модели), выставьте pos=False
    # в этом случае на выход будут поданы только леммы
    # По умолчанию знаки пунктуации вырезаются. Чтобы сохранить их, выставьте punct=True

    entities = {"PROPN"}
    named = False
    memory = []
    mem_case = None
    mem_number = None
    tagged_propn = []

    # обрабатываем текст, получаем результат в формате conllu:
    processed = pipeline.process(text)

    # пропускаем строки со служебной информацией:
    content = [line for line in processed.split("\n") if not line.startswith("#")]

    # извлекаем из обработанного текста леммы, тэги и морфологические характеристики
    tagged = [w.split("\t") for w in content if w]

    for t in tagged:
        if len(t) != 10:
            continue
        (word_id, token, lemma, pos, xpos, feats, head, deprel, deps, misc) = t
        token = clean_token(token, misc)
        lemma = clean_lemma(lemma, pos)
        if not lemma or not token:
            continue
        if pos in entities:
            if "|" not in feats:
                tagged_propn.append("%s_%s" % (lemma, pos))
                continue
            morph = {el.split("=")[0]: el.split("=")[1] for el in feats.split("|")}
            if "Case" not in morph or "Number" not in morph:
                tagged_propn.append("%s_%s" % (lemma, pos))
                continue
            if not named:
                named = True
                mem_case = morph["Case"]
                mem_number = morph["Number"]
            if morph["Case"] == mem_case and morph["Number"] == mem_number:
                memory.append(lemma)
                if "SpacesAfter=\\n" in misc or "SpacesAfter=\\s\\n" in misc:
                    named = False
                    past_lemma = "::".join(memory)
                    memory = []
                    tagged_propn.append(past_lemma + "_PROPN")
            else:
                named = False
                past_lemma = "::".join(memory)
                memory = []
                tagged_propn.append(past_lemma + "_PROPN")
                tagged_propn.append("%s_%s" % (lemma, pos))
        else:
            if not named:
                if pos == "NUM" and token.isdigit():  # Заменяем числа на xxxxx той же длины
                    lemma = num_replace(token)
                tagged_propn.append("%s_%s" % (lemma, pos))
            else:
                named = False
                past_lemma = "::".join(memory)
                memory = []
                tagged_propn.append(past_lemma + "_PROPN")
                tagged_propn.append("%s_%s" % (lemma, pos))

    if not keep_punct:
        tagged_propn = [word for word in tagged_propn if word.split("_")[1] != "PUNCT"]
    if not keep_pos:
        tagged_propn = [word.split("_")[0] for word in tagged_propn]
    return tagged_propn


model = Model.load("/content/udpipe_syntagrus.model")
process_pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

res = unify_sym(tweet.iloc[0, 3].strip())
output = process(process_pipeline, text=res)

<>:79: SyntaxWarning: invalid escape sequence '\|'
<>:157: SyntaxWarning: invalid escape sequence '\s'
<>:79: SyntaxWarning: invalid escape sequence '\|'
<>:157: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_370/2464298784.py:79: SyntaxWarning: invalid escape sequence '\|'
  '\t\n\r абвгдеёзжийклмнопрстуфхцчшщьыъэюяАБВГДЕЁЗЖИЙКЛМНОПРСТУФХЦЧШЩЬЫЪЭЮЯ,.[]{}()=+-−*&^%$#@!?~;:0123456789§/\|"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ '
/tmp/ipykernel_370/2464298784.py:157: SyntaxWarning: invalid escape sequence '\s'
  if "SpacesAfter=\\n" in misc or "SpacesAfter=\s\\n" in misc:


In [ ]:
output[-1], w2v_rus.get_vector(output[-1])

('тип_NOUN',
 array([ 2.3794374 , -1.2997365 ,  0.03844362, -2.0786328 , -0.70080346,
         0.9506083 , -1.5064685 ,  0.07744049, -1.265304  , -1.253626  ,
        -1.5600046 , -0.9778511 ,  0.32903767, -2.7029154 , -0.5932903 ,
        -0.09483349, -3.3329735 ,  0.36379638, -0.41304362, -2.0099683 ,
         2.2543058 , -0.60396576, -0.03874135, -2.8186388 ,  3.0297902 ,
         2.2236917 , -0.02021254,  0.05896755, -2.8987422 ,  0.19340906,
         0.66036344, -1.3866072 ,  4.2557077 ,  1.2211306 ,  0.03307049,
         0.2532471 ,  1.1391915 ,  0.51410913,  0.10419527, -1.7420535 ,
         1.7297152 ,  1.0689864 , -2.9722342 , -3.909114  ,  2.02839   ,
        -0.6308728 ,  1.3926637 ,  1.050261  ,  1.0938759 , -1.151322  ,
         2.0015092 ,  2.3261724 , -1.2071488 , -2.113711  ,  2.9428093 ,
        -0.46496394,  0.16155797, -2.6417034 , -1.1182822 , -0.8525918 ,
         1.9181919 ,  2.6665118 , -1.2361473 , -0.8567111 , -1.4306841 ,
        -1.2925864 ,  0.8352679 , -0.0

# 3. FastText

In [ ]:
ft = gensim.models.FastText(preprocessed, vector_size=300, min_count=2, window=3)

In [ ]:
ft.wv.get_vector("день")

array([ 1.74511239e-04,  4.15324146e-04, -1.11101195e-03, -5.48263306e-05,
        2.01808914e-04,  7.56583768e-06, -7.35740585e-04, -2.82628927e-04,
        4.53541463e-04, -4.67865437e-04,  5.98916376e-04, -2.80311884e-04,
       -4.62178286e-04,  6.37562654e-04,  4.41665979e-05, -7.18769326e-04,
        3.10315285e-04, -1.85799741e-04, -4.87660436e-04,  4.31913431e-05,
        1.28362211e-04, -1.91183761e-04, -4.29720472e-04, -2.92078301e-04,
        6.51413458e-04,  1.35826960e-03, -1.12468239e-04, -1.10594556e-07,
       -1.06851199e-04, -1.30054611e-03, -1.03450611e-05, -1.15229821e-04,
        4.49011044e-04, -4.22724297e-05,  8.16816930e-04, -2.08785947e-04,
        2.22074741e-05, -7.71374791e-04,  3.76590731e-04, -1.02139672e-03,
       -9.08587128e-04,  1.96955021e-04, -1.61980093e-03, -5.45494142e-04,
        8.20154150e-04, -7.96700537e-04,  3.81570775e-04,  1.07059386e-04,
       -4.74237284e-04, -4.36420087e-04, -8.06194439e-04, -4.09114553e-04,
       -3.40818631e-04,  

RusVectors скачиваем

In [ ]:
ft_rus = gensim.models.KeyedVectors.load("/content/ft/model.model")
ft_rus.get_vector("день_NOUN")

array([-0.07342825, -0.05677041,  0.09647455, -0.06419558,  0.07518704,
        0.29036632,  0.19890799,  0.13699473, -0.00049279,  0.18988532,
        0.10460955, -0.04094862, -0.18834533, -0.20590025,  0.05750467,
        0.04671513,  0.03870979, -0.09321839, -0.06431991,  0.01640876,
        0.13085824, -0.283238  , -0.2729816 , -0.01758899, -0.08311918,
        0.06204574,  0.01803116,  0.11200205, -0.2695526 , -0.01212822,
       -0.11670155, -0.10938901, -0.14384432,  0.05204165, -0.08536407,
       -0.15892613, -0.05165148, -0.05395661, -0.02446852, -0.28822023,
       -0.16177349,  0.05349648, -0.18739487,  0.08118755,  0.05733235,
       -0.03025746,  0.14820206,  0.00671048, -0.13802229, -0.0721654 ,
        0.04721965,  0.09663695,  0.19038868, -0.11077403, -0.06658518,
        0.05330818,  0.02309436, -0.07548448,  0.02828533, -0.13904195,
       -0.11255146, -0.01092859, -0.29169938,  0.0519538 ,  0.02408969,
       -0.10538361, -0.06363083,  0.05982324, -0.22058785, -0.01

# 4. Предобработка корпусов

In [ ]:
def get_sentence_vector(tokenizer, sentence, rus: bool = False):
    if rus:
        res = unify_sym(sentence.strip())
        sentence = process(process_pipeline, text=res)
    vecs = []
    for word in sentence:
        try:
            vec = tokenizer.get_vector(word)
        except Exception:
            vec = [0] * tokenizer.vector_size
        vecs.append(vec)
    return np.mean(vecs, axis=0)


w2v_feat = []
ft_feat = []
for sentence in tqdm(preprocessed):
    w2v_vec = get_sentence_vector(w2v.wv, sentence, False)
    ft_vec = get_sentence_vector(ft.wv, sentence, False)

    w2v_feat.append(w2v_vec)
    ft_feat.append(ft_vec)

NameError: name 'preprocessed' is not defined

In [ ]:
w2v_feat[0], ft_feat[0]

(array([-5.82076646e-02,  3.16417627e-02, -1.86736330e-01, -8.61544237e-02,
         2.05437024e-03,  1.27374493e-02, -1.74849145e-02, -9.17389542e-02,
        -3.87816317e-02,  6.25590459e-02, -1.16268672e-01, -1.72092114e-02,
         8.90828446e-02, -1.12219840e-01,  8.25002231e-03, -6.19500466e-02,
        -2.73369402e-02, -5.83901778e-02,  1.51729256e-01,  1.58480436e-01,
         5.32144494e-02, -1.10631831e-01, -1.25995994e-01, -6.08043484e-02,
        -6.98859543e-02, -6.19028183e-03, -6.62433580e-02,  3.86689343e-02,
         4.01623314e-04, -9.62192118e-02,  5.40007316e-02, -8.48154053e-02,
        -2.52858130e-03, -8.09424520e-02,  1.47980461e-02, -1.32011190e-01,
        -4.14534332e-03,  1.14139073e-01, -1.24016196e-01, -7.11454591e-03,
         5.80937676e-02, -9.68845561e-03,  3.21879871e-02,  1.63186844e-02,
        -2.17965096e-02,  7.40595832e-02,  2.62865834e-02, -2.63818130e-02,
         9.44681689e-02,  3.58518399e-02, -6.82313442e-02, -9.12571847e-02,
         3.9

In [ ]:
w2v_rus_feat = []
ft_rus_feat = []

for sentence in tqdm(sentences):
    w2v_vec = get_sentence_vector(w2v_rus, sentence, True)
    ft_vec = get_sentence_vector(ft_rus, sentence, True)

    w2v_rus_feat.append(w2v_vec)
    ft_rus_feat.append(ft_vec)

  0%|          | 0/226834 [00:00<?, ?it/s]

In [ ]:
w2v_rus_feat[0], ft_rus_feat[0]

(array([-0.43575193,  0.42378157,  0.11092638, -0.3002015 , -0.08364428,
         0.26326999, -0.28723749,  0.36835666, -0.17315424, -0.31838541,
        -0.37992662,  0.30048616,  0.14437263, -0.15476665, -0.04709676,
        -0.25253191,  0.13652901, -0.15668729,  0.70572549, -0.35492918,
         0.19930058,  0.02742164,  0.13016045, -0.30296813,  0.02382293,
         0.50375331,  0.37373713,  0.08879944, -0.37742974, -0.13773085,
         0.29118223, -0.34186966,  0.28417593,  0.11376459,  0.05931313,
        -0.35254112, -0.23591306, -0.25520092,  0.74000621, -0.53093828,
         0.32765292,  0.11263088, -0.40847945, -0.60210815, -0.28517054,
         0.11314276,  0.52987937,  0.41413649, -0.0196519 , -0.1888425 ,
         0.42852413,  0.00778393, -0.51769782,  0.14024242,  0.16528876,
        -0.28093468, -0.16753008, -0.6149508 , -0.21625194,  0.30909064,
        -0.14623553, -0.2225082 , -0.46114847, -0.52499764, -0.04624302,
        -0.44053774,  0.05792728, -0.37936863,  0.3

In [ ]:
# shapes = [arr.shape for arr in w2v_feat]
# print(set(shapes))
w2v_feat_fix = np.asarray([arr if arr.shape == (300,) else np.zeros(300) for arr in w2v_feat])
ft_feat_fix = np.asarray([arr if arr.shape == (300,) else np.zeros(300) for arr in ft_feat])

In [ ]:
# np.save("w2v.npy", np.asarray(w2v_feat_fix))
# np.save("ft.npy", np.asarray(ft_feat_fix))

np.save("w2v_rus.npy", np.asarray(w2v_rus_feat))
np.save("ft_rus.npy", np.asarray(ft_rus_feat))

In [ ]:
assert np.allclose(np.load("w2v.npy"), np.asarray(w2v_feat_fix))
assert np.allclose(np.load("ft.npy"), np.asarray(ft_feat_fix))

assert np.allclose(np.load("w2v_rus.npy"), np.asarray(w2v_rus_feat))
assert np.allclose(np.load("ft_rus.npy"), np.asarray(ft_rus_feat))